# 결측치 대체 — R 데이터 전처리
## 개요
친환경 슬래그 시멘트 공정 실험 데이터를 대상으로, 엑셀 로드·열 선택·독립변수 결측치 대체(MICE, missForest, 1D-CNN)까지 수행하는 1단계 전처리 노트북이다.

데이터는 불완전(MCAR에 가까운 구조)하며, 연속형·범주형이 혼재한 혼합형 공정 변수를 다룬다.

## 파일 역할
- `sources/` 엑셀 입력 → `results/` 및 분석 스탬프 디렉터리에 중간·최종 산출물 저장
- `custom_library/` 헬퍼(R kernel, 한글 설정, MICE 집계, 엑셀 하이라이트) 호출

## 작성 이력
- 최초 작성: 2025-07-01

## Step 00: 전처리

### Kernel 생성 및 연결

In [31]:
# kernel 설정 및 확인
source("./custom_library/fn_01_IRkernel_Setup.R")

irkernel_setup("missing-imputation", "R(MissingImputation)")


### Visual Studio Code 환경에서 한글 적용이 가능하도록 설정

In [32]:
source("custom_library/fn_01_korean_setting.R")
korean_setting()

한국어 로케일 설정 완료: ko_KR.UTF-8
ggplot2 테마 설정 완료

=== 한글 출력 설정 확인 ===
현재 로케일: ko_KR.UTF-8/ko_KR.UTF-8/ko_KR.UTF-8/C/ko_KR.UTF-8/C 
사용 폰트: D2Coding 
인코딩: UTF-8 
ggplot2 로드 상태: TRUE 
ggplot2 테마 설정 상태: TRUE 

=== 세션 정보 ===
현재 시간: 2025-07-15 18:11:39 
R 버전: R version 4.3.2 (2023-10-31) 
R version 4.3.2 (2023-10-31)
Platform: aarch64-apple-darwin20.0.0 (64-bit)
Running under: macOS 15.5

Matrix products: default
BLAS/LAPACK: /opt/anaconda3/lib/libopenblas.0.dylib;  LAPACK version 3.12.0

locale:
[1] ko_KR.UTF-8/ko_KR.UTF-8/ko_KR.UTF-8/C/ko_KR.UTF-8/C

time zone: Asia/Seoul
tzcode source: system (macOS)

attached base packages:
[1] stats     graphics  grDevices datasets  utils     methods   base     

other attached packages:
 [1] tensorflow_2.16.0 missForest_1.5    mice_3.18.0       reticulate_1.42.0
 [5] broom_1.0.7       reshape2_1.4.4    lubridate_1.9.4   openxlsx_4.2.8   
 [9] tidyr_1.3.1       dplyr_1.1.4       pacman_0.5.1      ggplot2_3.5.1    
[13] IRkernel_1.3.2   

loaded via a namespa

In [33]:
# =============================================================================
# 다중 출력 예측 모델링을 위한 종합 분석 코드
# =============================================================================

# 필요한 패키지 설치 및 로드
if (!require("pacman")) install.packages("pacman", quietly = TRUE)
pacman::p_load(
  # 기본 데이터 처리
  dplyr, tidyr, openxlsx, lubridate, reshape2, broom, reticulate
)


In [34]:
# 시각화 설정
options(repr.plot.width=12, repr.plot.height=8)
options(repr.plot.res = 300)


### 현재 작업 환경의 확인

In [35]:
#SELECTED_FILE <- "CSM_25617_JHBae"
SELECTED_FILE <- "Imputation_Test"
SETTING_FILE  <- "dirsetting.txt"

current_dir   <- getwd()
WORK_DATE     <- format(Sys.time(), "%Y%m%d")
WORK_TIME     <- format(Sys.time(), "%H%M%S")

source("custom_library/fn_01_directory_setting.R")
directory_setting(SELECTED_FILE, current_dir, WORK_DATE, WORK_TIME)
    # 사용된 디렉토리 저장 파일: dirsetting.txt
    # 1열: WORK_DIR
    # 2열: RSLT_DIR
    # 3열: ANAL_DIR
    # 4열: SRC_FILE
    # 5열: EXCEL_FILE
    # 6열: PDF_FILE

현재 작업 경로:     /Users/jhbae/Documents/Code/CSM_2506 

1열 - 분석 대상 파일경로: /Users/jhbae/Documents/Code/CSM_2506/sources 
2열 - 분석 결과 파일경로: /Users/jhbae/Documents/Code/CSM_2506/results 
3열 - 분석 진행 파일경로: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715 

4열 - 분석 대상 파일    : /Users/jhbae/Documents/Code/CSM_2506/sources/Imputation_Test.xlsx 
5열 - 분석 진행 엑셀파일: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 
6열 - 분석 진행 PDF 파일: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 
dirsetting.txt 파일이 생성되었습니다:  /Users/jhbae/Documents/Code/CSM_2506/dirsetting.txt 


### 기타 전역 함수 (custom function) 로드

In [36]:
# 결측치 색상 처리 및 엑셀 파일 시트 저장 함수
source("custom_library/fn_01_write_excel.R")
# write_excel(data, file_path, sheet_name, null_color, null_text_color)

## Step 01: 데이터 로드

### srcData: 분석 대상 엑셀 파일의 첫번 째 탭에서 분석 영역만 취합

In [8]:
# dirsetting.txt 파일의 4번째 라인을 읽어와 SRC_FILE 변수에 저장
SRC_FILE <- readLines(SETTING_FILE, warn = FALSE)[4]


# =============================================================================
# Step 1: 데이터 로드 및 전처리
# =============================================================================

cat("Step 1: 데이터 로드 및 전처리\n")
cat("----------------------------------------\n")

# 데이터 읽기
srcData <- read.xlsx(
  xlsxFile = SRC_FILE,
  sheet    = 1,               # "Sheet1", 정확한 시트명을 모르면, 순서를 번호로 (1, 2, 3...) 입력해도 됨
  startRow = 3,
  colNames = TRUE,
  skipEmptyRows = TRUE,       # 빈 행 건너뛰기
  skipEmptyCols = TRUE        # 빈 열 건너뛰기
)

cat("원본 데이터 차원(행x열)    :", dim(srcData), "\n")

# 불필요한 열 제거
srcData <- srcData %>%
  select(-1:-3, -5:-6) %>%    # 실험일자, 목적, 차수, RMth, 번호 제거
  select(-length(.))          # 마지막 열(비고) 제거

# 모든 값이 0이거나 NA인 열 제거
srcData <- srcData %>% 
  select_if(~ !(all(is.na(.) | . == 0)))

cat("전처리 후 데이터 차원(행x열):", dim(srcData), "\n")
cat("결측치 현황:\n")
print(colSums(is.na(srcData)))

Step 1: 데이터 로드 및 전처리
----------------------------------------
원본 데이터 차원(행x열)    : 451 70 
전처리 후 데이터 차원(행x열): 451 63 
결측치 현황:
       Index       Class1       Class2          CaO        Al2O3         SiO2 
           0            0            0            0            0            0 
       Fe2O3          SO3          MgO          etc           FM        MeanV 
           0            0            0            0           10           10 
     Density        Blain           HG           AG           DG      Gyp_per 
          10           10           10           10           10           10 
   CA_Before    GS_Before    JA_Before    CS_Before Delay_Bf_per     CA_after 
         261          261          261          261          261          111 
    BA_after     CS_after     SC_after Delay_Af_per    ZS_Before   ZS_Milling 
         111          111          111          111          277          277 
    ZS_After   H2O2_After    IS_Before  FeCl2_After    FeS_After     I_Before 
      

In [9]:
head(srcData)

,Index,Class1,Class2,CaO,Al2O3,SiO2,Fe2O3,SO3,MgO,etc,...,H2S,Flow,H3,H4,D1,D3,D28,D91,D180,D365
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,231023-02,현대,시멘트,50.07,29.69,9.64,2.75,0.70,3.73,3.42,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,230411-01,세아,시멘트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,...,137,190.0,56.1,NA,66.1,71.2,79.7,NA,NA,NA
3,230406-02,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,...,NA,155.0,1.1,NA,1.8,2.6,2.3,NA,NA,NA
4,230316-05,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,...,NA,152.5,0.3,NA,1.3,1.5,1.5,NA,NA,NA
5,210713-03,세아,시멘트,49.07,29.81,7.83,1.94,3.44,6.36,1.55,...,137,225.0,43.0,NA,64.0,66.0,64.7,NA,NA,NA
6,240708-08,세아,시멘트,47.46,27.91,8.34,0.68,4.12,6.10,5.39,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [37]:
# srcData를 엑셀 파일로 저장
output_file <- readLines(SETTING_FILE, warn = FALSE)[5]


# 엑셀 파일로 저장
write.xlsx(
  x = srcData,
  file = output_file,
  sheetName = "srcData",
  rowNames = FALSE,
  colNames = TRUE
)


cat("소스 데이터가 저장되었습니다:", output_file, "\n")


소스 데이터가 저장되었습니다: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 


### data_bf_Imputation: 결측치 대상 영역만 선택하고, 결측치 처리 전 데이터 저장

In [38]:
# 독립변수와 종속변수를 분리
src_In_Var <- srcData %>% select(1:50)
src_Out_Var <- srcData %>% select(1, 51:length(.))

# 독립변수 중 결측치 처리 대상 칼럼 선정
data_bf_Imputation <- src_In_Var %>% select(1:18, 50)

head(data_bf_Imputation)

,Index,Class1,Class2,CaO,Al2O3,SiO2,Fe2O3,SO3,MgO,etc,FM,MeanV,Density,Blain,HG,AG,DG,Gyp_per,WC_per
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,231023-02,현대,시멘트,50.07,29.69,9.64,2.75,0.70,3.73,3.42,3.50,7.394,3.10,6286,1,0,0,20,0.50
2,230411-01,세아,시멘트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.50
3,230406-02,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
4,230316-05,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
5,210713-03,세아,시멘트,49.07,29.81,7.83,1.94,3.44,6.36,1.55,3.18,5.450,2.91,5994,1,0,0,25,0.50
6,240708-08,세아,시멘트,47.46,27.91,8.34,0.68,4.12,6.10,5.39,3.23,7.394,2.97,5998,1,0,0,25,0.40


In [39]:
# 원본 독립변수 저장
# 기존 엑셀 파일에 시트 추가
write_excel(
  data = src_In_Var,
  file = output_file,
  sheet_name = "원본독립변수"
)

# 원본 종속변수 저장
write_excel(
  data = src_Out_Var,
  file = output_file,
  sheet_name = "원본종속변수"
)

# 결측치 처리 대상 데이터 저장
write_excel(
  data = data_bf_Imputation,
  file_path = output_file,
  sheet_name = "Before_Imputation",
  null_color = "#FFCCCC",      # 진한 분홍색
  null_text_color = "#FF0000"  # 붉은 텍스트
)

원본독립변수 의 NULL 값 셀 7101 개를 #FFFFFF 색으로 하이라이트했습니다.
Excel 파일 저장 완료: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 
원본종속변수 의 NULL 값 셀 3916 개를 #FFFFFF 색으로 하이라이트했습니다.
Excel 파일 저장 완료: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 
Before_Imputation 의 NULL 값 셀 90 개를 #FFCCCC 색으로 하이라이트했습니다.
Excel 파일 저장 완료: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 


## Step 02. 연쇄방정식(MICE)을 이용한 결측치 처리

### 권장사항
| 분석 목적 | 권장 방법 | 이유 |
|:--- | :---: | :--- |
| 탐색적 데이터 분석 | 평균값 혹은 단일 세트 | 빠르고 직관적 |
| 시각화 및 기술통계 | 평균값 또는 중앙값 | 이상치가 많다면 중앙값 |
| 통계적 추론 | 다중 대치 분석 | 불확실성 고려 |
| 머신러닝 모델링 | 평균값 또는 단일 세트 | 모델 성능에 따라 선택 |

### 함수 사용 방법
`calculate_mice(imputed_mice, result_type, set_number)`
- imputed_mice : mice를 통해 결측치 처리를 한 결과
- result_type : 여러 세트의 결측치 처리 결과를 후처리 하는 방법
  - result_type = 1 : 특정 세트의 결과값 활용 (반드시 set_number 설정 필요)
  - result_type = 2 : 중앙값 활용
  - result_type = 3 : 최빈값 활용 (범주용 사용에 적합)
  - result_type = 4 : 평균값 활용
  - result_type = 5 : 모든 세트의 결과를 모두 (테스트 결과 확인용으로만 사용)
- set_number : 단일 대치값 사용 시, 몇 번째 세트값을 활용할지 설정 (최대 m의 값을 넘을 수 없음) 

In [40]:
# 필요한 라이브러리 설치 및 로드
source("custom_library/fn_01_calculate_mice.R")

# 1. 결측치 대치 처리
imputed_mice <- mice(data_bf_Imputation, m = 10, method = "pmm", maxit = 10, seed = 123)

# 2. 결측치 대치 결과 확인
data_af_mice <- calculate_mice(imputed_mice, result_type = 4)

# 4. 결과 확인
summary(data_af_mice)
anyNA(data_af_mice)


 iter imp variable
  1   1  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   2  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   3  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   4  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   5  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   6  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   7  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   8  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   9  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  1   10  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   1  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   2  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   3  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   4  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   5  FM  MeanV  Density  Blain  HG  AG  DG  Gyp_per  WC_per
  2   6  FM  MeanV 

Warning message:
“Number of logged events: 903”


    Index              Class1             Class2               CaO       
 Length:451         Length:451         Length:451         Min.   :43.95  
 Class :character   Class :character   Class :character   1st Qu.:47.46  
 Mode  :character   Mode  :character   Mode  :character   Median :49.10  
                                                          Mean   :48.72  
                                                          3rd Qu.:50.10  
                                                          Max.   :53.40  
     Al2O3            SiO2            Fe2O3             SO3       
 Min.   :19.20   Min.   : 7.780   Min.   : 0.550   Min.   :0.050  
 1st Qu.:27.28   1st Qu.: 8.200   1st Qu.: 0.680   1st Qu.:0.640  
 Median :27.91   Median : 8.340   Median : 1.180   Median :3.900  
 Mean   :28.01   Mean   : 9.343   Mean   : 2.182   Mean   :2.687  
 3rd Qu.:29.81   3rd Qu.:11.340   3rd Qu.: 1.940   3rd Qu.:4.120  
 Max.   :31.20   Max.   :12.700   Max.   :11.800   Max.   :4.700  
      MgO    

[1] FALSE

In [41]:
head(data_af_mice)

,Index,Class1,Class2,CaO,Al2O3,SiO2,Fe2O3,SO3,MgO,etc,FM,MeanV,Density,Blain,HG,AG,DG,Gyp_per,WC_per
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,231023-02,현대,시멘트,50.07,29.69,9.64,2.75,0.70,3.73,3.42,3.50,7.394,3.10,6286,1,0,0,20,0.50
2,230411-01,세아,시멘트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.50
3,230406-02,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
4,230316-05,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
5,210713-03,세아,시멘트,49.07,29.81,7.83,1.94,3.44,6.36,1.55,3.18,5.450,2.91,5994,1,0,0,25,0.50
6,240708-08,세아,시멘트,47.46,27.91,8.34,0.68,4.12,6.10,5.39,3.23,7.394,2.97,5998,1,0,0,25,0.40


In [42]:
# mice 결측치 처리 후 데이터 저장
write_excel(
  data = data_af_mice,
  file_path = output_file,
  sheet_name = "After_MICE",
  null_color = "#FF6666",      # 진한 분홍색
  null_text_color = "#FFFFFF"  # 흰색 텍스트
)

After_MICE 의 NULL 값이 발견되지 않았습니다.
Excel 파일 저장 완료: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 


## Step 03. missForest를 이용한 결측치 처리

다음에 실행하는 missForest를 활용한 데이터 처리에, 다음과 같은 후속 처리를 고민할 필요가 있다. (20250702)
1. 추정된 결측치의 소수점 이하 처리
2. HG, AG, DG 등의 One-hot Encoding 처리된 값의 정리 (가장 큰 값만 1, 나머지는 0)

In [43]:
# 1. 패키지 설치 및 로드
if (!require(missForest, quietly = TRUE)) {
  install.packages("missForest")
}
library(missForest)

# 2. 처음 열을 분리하여, 별도 보관
## Index 칼럼을 별도로 분리
index_column <- data_bf_Imputation$Index

# Index 컬럼을 제외한 데이터
data_without_index <- data_bf_Imputation[, -1]

# 3. 처음열이 제외된 데이터 셋에서, 문자형 컬럼을 모두 factor로 변환
## 문자형 컬럼들을 factor로 변환
data_without_index <- data_without_index %>%
  mutate(across(where(is.character), as.factor))

# 4. 결측치 대치
imputed_result <- missForest(data_without_index, 
                             maxiter = 10, 
                             ntree = 100, 
                             variablewise = TRUE, 
                             verbose = TRUE)

# 5. 대치된 데이터 추출
data_without_index <- imputed_result$ximp

# 6. 결과 확인
summary(data_without_index)
anyNA(data_without_index)

# 7. 분리한 처음열과 결합 후 저장
## Index 컬럼을 다시 추가
data_af_missForest <- data_without_index %>%
  mutate(Index = index_column) %>%
  select(Index, everything())



  missForest iteration 1 in progress...

Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”


done!
    estimated error(s): 0 0 0 0 0 0 0 0 0 0.08554339 0.180173 8.692097e-05 3533.741 0.007278295 0.008228585 0.006964908 16.77515 0.002038456 
    difference(s): 7.337301e-05 0 
    time: 0.299 seconds

  missForest iteration 2 in progress...

Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”


done!
    estimated error(s): 0 0 0 0 0 0 0 0 0 0.08413932 0.1673846 7.943062e-05 2552.811 0.006402199 0.007148183 0.006413385 16.87594 0.002044069 
    difference(s): 8.542505e-08 0 
    time: 0.288 seconds

  missForest iteration 3 in progress...

Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”
Warning message in randomForest.default(x = obsX, y = obsY, ntree = ntree, mtry = mtry, :
“The response has five or fewer unique values.  Are you sure you want to do regression?”


done!
    estimated error(s): 0 0 0 0 0 0 0 0 0 0.08171487 0.1662876 5.81368e-05 2228.086 0.006850697 0.009329656 0.006932008 15.86905 0.00201807 
    difference(s): 1.421837e-07 0 
    time: 0.285 seconds



  Class1             Class2         CaO            Al2O3            SiO2       
 세아:248   급결재      :  2   Min.   :43.95   Min.   :19.20   Min.   : 7.780  
 현대:203   기포콘크리트: 17   1st Qu.:47.46   1st Qu.:27.28   1st Qu.: 8.200  
            시멘트      :432   Median :49.10   Median :27.91   Median : 8.340  
                               Mean   :48.72   Mean   :28.01   Mean   : 9.343  
                               3rd Qu.:50.10   3rd Qu.:29.81   3rd Qu.:11.340  
                               Max.   :53.40   Max.   :31.20   Max.   :12.700  
     Fe2O3             SO3             MgO             etc       
 Min.   : 0.550   Min.   :0.050   Min.   :2.950   Min.   :1.110  
 1st Qu.: 0.680   1st Qu.:0.640   1st Qu.:5.040   1st Qu.:1.980  
 Median : 1.180   Median :3.900   Median :6.100   Median :2.840  
 Mean   : 2.182   Mean   :2.687   Mean   :5.838   Mean   :3.216  
 3rd Qu.: 1.940   3rd Qu.:4.120   3rd Qu.:6.520   3rd Qu.:4.320  
 Max.   :11.800   Max.   :4.700   Max.   :9.050   Max.   :8.

[1] FALSE

In [44]:
head(data_af_missForest)

,Index,Class1,Class2,CaO,Al2O3,SiO2,Fe2O3,SO3,MgO,etc,FM,MeanV,Density,Blain,HG,AG,DG,Gyp_per,WC_per
,<chr>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,231023-02,현대,시멘트,50.07,29.69,9.64,2.75,0.70,3.73,3.42,3.50,7.394,3.10,6286,1,0,0,20,0.50
2,230411-01,세아,시멘트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.50
3,230406-02,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
4,230316-05,세아,기포콘크리트,46.65,30.73,7.78,0.56,4.70,7.60,1.98,2.84,5.120,2.93,6152,1,0,0,25,0.35
5,210713-03,세아,시멘트,49.07,29.81,7.83,1.94,3.44,6.36,1.55,3.18,5.450,2.91,5994,1,0,0,25,0.50
6,240708-08,세아,시멘트,47.46,27.91,8.34,0.68,4.12,6.10,5.39,3.23,7.394,2.97,5998,1,0,0,25,0.40


In [45]:
# mice 결측치 처리 후 데이터 저장
write_excel(
  data = data_af_missForest,
  file_path = output_file,
  sheet_name = "After_missForest",
  null_color = "#FF6666",      # 진한 분홍색
  null_text_color = "#FFFFFF"  # 흰색 텍스트
)

After_missForest 의 NULL 값이 발견되지 않았습니다.
Excel 파일 저장 완료: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.xlsx 


## Step 04. 1D-CNN을 활용한 결측치 처리

In [20]:
# 문제 해결을 위한 확인/조치
library(reticulate)

# 파이썬 환경 확인
py_config()

reticulate::py_module_available("tensorflow")

python:         /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/iVhX6JXLXk0R_7v5GFDy3/bin/python3
libpython:      /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/python/cpython-3.11.13-macos-aarch64-none/lib/libpython3.11.dylib
pythonhome:     /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/iVhX6JXLXk0R_7v5GFDy3:/Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/iVhX6JXLXk0R_7v5GFDy3
virtualenv:     /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/iVhX6JXLXk0R_7v5GFDy3/bin/activate_this.py
version:        3.11.13 (main, Jun 29 2025, 16:05:00) [Clang 20.1.4 ]
numpy:          /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/iVhX6JXLXk0R_7v5GFDy3/lib/python3.11/site-packages/numpy
numpy_version:  2.3.1

NOTE: Python version was forced by py_require()

[1] FALSE

In [21]:
# 관련 패키지 설정
if (!require(keras, quietly = TRUE)) install.packages("keras")
if (!require(tensorflow, quietly = TRUE)) install.packages("tensorflow")
library(keras)
install_tensorflow(version = "2.15.0", extra_packages = "tensorflow")    # CPU 버전으로 설치
library(tensorflow)
install_keras()



Warning message:
“ModuleNotFoundError: No module named 'tensorflow'
Run `reticulate::py_last_error()` for details.
Restart the R session and load the tensorflow R package before reticulate has initialized Python, or ensure reticulate initialized a Python installation where the tensorflow module is installed.”


The following package(s) will be installed:
- keras [2.15.0]
These packages will be installed into "~/Documents/Code/CSM_2506/renv/library/R-4.3/aarch64-apple-darwin20.0.0".

# Installing packages --------------------------------------------------------
- Installing keras ...                          OK [linked from cache]


ERROR: Error: package or namespace load failed for ‘keras’:
 .onLoad failed in loadNamespace() for 'keras', details:
  call: py_module_import(module, convert = convert)
  error: ModuleNotFoundError: No module named 'tensorflow'
Run `reticulate::py_last_error()` for details.


In [22]:
# GPU 사용 여부 설정 (TRUE/FALSE로 변경 가능)
use_gpu <- FALSE

if (use_gpu) {
  # GPU 설정
  gpus <- tf$config$experimental$list_physical_devices('GPU')
  if (length(gpus) > 0) {
    tf$config$experimental$set_memory_growth(gpus[[1]], TRUE)
    cat("GPU 사용 설정 완료\n")
  } else {
    cat("GPU를 찾을 수 없습니다. CPU를 사용합니다.\n")
  }
} else {
  # CPU 강제 사용
  Sys.setenv(CUDA_VISIBLE_DEVICES = -1)
  cat("CPU 사용 설정 완료\n")
}

CPU 사용 설정 완료


In [23]:
# 3. 데이터 분리
Cdata_with_Index <- data_bf_Imputation[, 1:3, drop = FALSE]
Cdata_without_Index <- data_bf_Imputation[, -(1:3), drop = FALSE]

# 4. 1D-CNN 결측치 대체 함수
impute_col_with_1dcnn <- function(df, target_col) {
  if (all(!is.na(df[[target_col]]))) return(df[[target_col]])
  
  train_idx <- which(!is.na(df[[target_col]]))
  test_idx  <- which(is.na(df[[target_col]]))
  
  input_cols <- setdiff(names(df), target_col)
  # 문자형 컬럼 제거
  is_char <- sapply(df[, input_cols, drop = FALSE], is.character)
  input_cols <- input_cols[!is_char]
  
  X_train <- as.matrix(df[train_idx, input_cols, drop = FALSE])
  Y_train <- as.numeric(df[train_idx, target_col])
  Y_train <- as.matrix(Y_train)
  X_test  <- as.matrix(df[test_idx, input_cols, drop = FALSE])
  
  input_dim <- as.integer(ncol(X_train))
  n_train <- as.integer(nrow(X_train))
  n_test  <- as.integer(nrow(X_test))
  
  # numpy array로 직접 변환
  np <- reticulate::import("numpy", convert = FALSE)
  n_train <- as.integer(nrow(X_train))
  input_dim <- as.integer(ncol(X_train))
 n_test <- as.integer(nrow(X_test))

  X_train_cnn <- np$array(X_train)$astype("float32")$reshape(
    as.integer(n_train), as.integer(input_dim), as.integer(1)
  )
  X_test_cnn  <- np$array(X_test)$astype("float32")$reshape(
    as.integer(n_test), as.integer(input_dim), as.integer(1)
  )
  
  model <- keras_model_sequential()
  model$add(layer_conv_1d(filters = 32, kernel_size = 2, activation = "relu", input_shape = c(input_dim, 1)))
  model$add(layer_flatten())
  model$add(layer_dense(units = 64, activation = "relu"))
  model$add(layer_dense(units = 1))
  
  model$compile(
    loss = "mse",
    optimizer = "adam"
  )
  
  model$fit(
    x = X_train_cnn, y = Y_train,
    epochs = 50,
    batch_size = 16,
    validation_split = 0.2,
    verbose = 0
  )
  
  Y_pred <- model$predict(X_test_cnn)
  result <- df[[target_col]]
  result[test_idx] <- as.vector(Y_pred)
  return(result)
}

# 5. 모든 칼럼에 대해 결측치 대체 반복
Cdata_imputed <- Cdata_without_Index
for (col in names(Cdata_imputed)) {
  cat("1D-CNN으로 결측치 대체 중:", col, "\n")
  Cdata_imputed[[col]] <- impute_col_with_1dcnn(Cdata_imputed, col)
}

# 6. 인덱스와 결측치 대체된 데이터 결합
data_af_1DCNN <- cbind(Cdata_with_Index, Cdata_imputed)

# 7. 결과 확인
cat("결측치 대체 완료! data_af_1DCNN의 결측치 개수:", sum(is.na(data_af_1DCNN)), "\n")


1D-CNN으로 결측치 대체 중: CaO 
1D-CNN으로 결측치 대체 중: Al2O3 
1D-CNN으로 결측치 대체 중: SiO2 
1D-CNN으로 결측치 대체 중: Fe2O3 
1D-CNN으로 결측치 대체 중: SO3 
1D-CNN으로 결측치 대체 중: MgO 
1D-CNN으로 결측치 대체 중: etc 
1D-CNN으로 결측치 대체 중: FM 


ERROR: Error in keras_model_sequential(): could not find function "keras_model_sequential"


In [24]:
str(X_train_cnn)
str(dim(X_train_cnn))



ERROR: Error: object 'X_train_cnn' not found


In [111]:
reticulate::py_last_error()



── Python Exception Message ────────────────────────────────────────────────────



── R Traceback ─────────────────────────────────────────────────────────────────



TypeError: 'float' object cannot be interpreted as an integer
     ▆
  1. ├─IRkernel::main()
  2. │ └─kernel$run() at IRkernel/R/main.r:15:5
  3. │   └─IRkernel (local) handle_shell() at IRkernel/R/kernel.r:419:13
  4. │     └─executor$execute(msg) at IRkernel/R/kernel.r:120:5
  5. │       ├─base::tryCatch(...) at IRkernel/R/execution.r:312:5
  6. │       │ └─base (local) tryCatchList(expr, classes, parentenv, handlers)
  7. │       │   ├─base (local) tryCatchOne(...)
  8. │       │   │ └─base (local) doTryCatch(return(expr), name, parentenv, handler)
  9. │       │   └─base (local) tryCatchList(expr, names[-nh], parentenv, handlers[-nh])
 10. │       │     └─base (local) tryCatchOne(expr, names, parentenv, handlers[[1L]])
 11. │       │       └─base (local) doTryCatch(return(expr), name, parentenv, handler)
 12. │       └─evaluate::evaluate(...) at IRkernel/R/execution.r:312:5
 13. │         ├─base::withRestarts(...) at evaluate/R/evaluate.R:146:5
 14. │         │ └─base (local) withR

In [124]:
cat("R version:", R.version.string, "\n")
cat("reticulate version:", as.character(packageVersion("reticulate")), "\n")
cat("keras (R) version:", as.character(packageVersion("keras")), "\n")
cat("tensorflow (R) version:", as.character(packageVersion("tensorflow")), "\n")

library(reticulate)
cat("Python path:", py_config()$python, "\n")
cat("Python version:", as.character(py_config()$version), "\n")

py_run_string("import tensorflow as tf; print('Python tensorflow version:', tf.__version__)")
py_run_string("import keras; print('Python keras version:', keras.__version__)")

R version: R version 4.3.2 (2023-10-31) 
reticulate version: 1.42.0 
keras (R) version: 2.15.0 
tensorflow (R) version: 2.16.0 
Python path: /Users/jhbae/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/f4TqLW_R_PwlEA_0ztRy-/bin/python3 
Python version: 3.11 


In [121]:
packageVersion("reticulate")

ERROR: Error in nchar(text_repr): invalid multibyte string, element 1


## Step 06. 결측치 대체 후 시각화

In [58]:
# 수정된 버전 사용
source("enhanced_distribution_plot_4graphs_fixed.R")


Columns to analyze: FM, MeanV, Density, Blain, HG, AG, DG, Gyp_per, WC_per 
Number of columns: 9 
PDF filename: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.pdf 
Generating all individual plots...
Processing: FM 
Processing: MeanV 
Processing: Density 
Processing: Blain 
Processing: HG 
Processing: AG 
Processing: DG 
Processing: Gyp_per 
Processing: WC_per 
All plots generated!
Saving to PDF with print margins (A4 Landscape)...
Creating new PDF file: /Users/jhbae/Documents/Code/CSM_2506/results/analysis_20250715/SourceData181149.pdf 
Saving page with plots: FM_mice, FM_missForest, MeanV_mice, MeanV_missForest 


ERROR: Error in as.unit(e2): object is not coercible to a unit
